In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re

import nltk
from nltk.corpus import stopwords
# nltk.download('stopwords')
# nltk.download('punkt')

STOPWORDS = set(stopwords.words('english'))

## Removing stopwords and saving adjectives

In [3]:
def save(filename, dep_type, inferred, pos):
    if inferred:
        inferred_type = "_inferred"
    else: 
        inferred_type = ""

    df= pd.read_json(filename, lines=True)

    df_no_stopwords = df[~df["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

    df_adjs = df_no_stopwords[df_no_stopwords["majority_upos"] == pos]
    
    df_adjs.to_json(
        f"verbs/verbs_{dep_type}{inferred_type}.ndjson",
        orient="records",
        lines=True
    )
    
    


### not inferred (includes chatgpt as model)

In [ ]:
save("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", "direct", False, "ADJ")

save("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", "onehop", False, "ADJ")

In [4]:
save("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", "direct", False, "VERB")

save("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", "onehop", False, "VERB")

### inferred (chatgpt entries distributed among other models based on model_inferred_temporal)

In [ ]:
save("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos_inferred.ndjson", "direct", True, "ADJ")

save("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos_inferred.ndjson", "onehop", True, "ADJ")


In [5]:
save("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos_inferred.ndjson", "direct", True, "VERB")

save("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos_inferred.ndjson", "onehop", True, "VERB")

## Saving adjectives by model

In [23]:
def save_adjectives_by_model(filename, dep_type, inferred):
    

    df= pd.read_json(filename, lines=True)

    df_no_stopwords = df[~df["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

    df_adjs = df_no_stopwords[df_no_stopwords["majority_upos"] == "ADJ"]

    unique_models = df_adjs["majority_model"].unique()
    for model in unique_models:
        df_model = df_adjs[df_adjs["majority_model"] == model].sort_values(by="total_count", ascending=False)

        model_type = model.replace("-", "_").replace(".", "_")

        if inferred:
            df_model.to_json(f"adjectives/by_model/{dep_type}/inferred/{model_type}_adjectives_{dep_type}_inferred.ndjson", orient="records", indent=4, lines=True) 
        else:
            df_model.to_json(f"adjectives/by_model/{dep_type}/not_inferred/{model_type}_adjectives_{dep_type}.ndjson", orient="records", indent=4, lines=True) 

### by model, not inferred

In [27]:
save_adjectives_by_model("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos.ndjson", "direct", False)

save_adjectives_by_model("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", "onehop", False)

### by model, inferred

In [28]:
save_adjectives_by_model("deps/big_corpus/direct/big_corpus_direct_cleaned_deps_mostfreqpos_inferred.ndjson", "direct", True)

save_adjectives_by_model("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos_inferred.ndjson", "onehop", True)

Visualizing top 10 total words, adjectives, verbs, and 
myself, yourself, himself, herself, itself, ourselves, yourselves, themselves
mine, yours, his, hers, ours, yours, theirs.

In [ ]:
PRONOUNS = ['i', 'me', 'myself', 'mine',
            'you', 'yourself' 'yours',
            'he', 'him', 'himself', 'his',
            'she', 'her', 'herself', 'hers',
            'it', 'itself', 'its',
            'we', 'us', 'ourselves', 'ours'
            'they', 'them', 'themselves', 'yourselves', 'theirs'
]


In [23]:
df = pd.read_json("deps/big_corpus/onehop/big_corpus_onehop_cleaned_deps_mostfreqpos.ndjson", encoding='utf-8', lines=True)



df_results = pd.DataFrame()
df_no_stopwords = df[~df["word"].str.lower().isin(STOPWORDS)].sort_values(by="total_count", ascending=False)

df_results["Top Overall"] = df_no_stopwords.sort_values(by="total_count", ascending=False)["word"].head(10).values
df_results["Top Overall Count"] = df_no_stopwords.sort_values(by="total_count", ascending=False)["total_count"].head(10).values

df_results["Top ADJ"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "ADJ"].sort_values(by="total_count", ascending=False)["word"].head(10).values
df_results["Top ADJ Count"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "ADJ"].sort_values(by="total_count", ascending=False)["total_count"].head(10).values

df_results["Top VERB"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "VERB"].sort_values(by="total_count", ascending=False)["word"].head(10).values
df_results["Top VERB Count"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "VERB"].sort_values(by="total_count", ascending=False)["total_count"].head(10).values

df_results["Top NOUN"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "NOUN"].sort_values(by="total_count", ascending=False)["word"].head(10).values
df_results["Top NOUN Count"] = df_no_stopwords[df_no_stopwords["majority_upos"] == "NOUN"].sort_values(by="total_count", ascending=False)["total_count"].head(10).values


In [24]:
df_results


,Top Overall,Top Overall Count,Top ADJ,Top ADJ Count,Top VERB,Top VERB Count,Top NOUN,Top NOUN Count
0,use,185360,good,36965,use,185360,model,36268
1,ask,89965,prompt,25209,ask,89965,ai,23715
2,get,59853,free,10510,get,59853,url,19532
3,make,53502,able,10094,make,53502,answer,18107
4,like,47616,new,8924,say,44340,way,17672
5,say,44340,bad,8213,write,38838,version,16767
6,write,38838,sure,7236,think,36838,thing,16107
7,good,36965,great,5420,give,34837,response,16092
8,think,36838,right,4755,tell,31574,people,14623
9,model,36268,available,4425,know,29302,access,14405


In [51]:
df_pronouns = (
    df[df["word"].str.lower().isin(PRONOUNS)]
    .assign(word_lower=df["word"].str.lower())
    .groupby("word_lower", as_index=False)["total_count"]
    .sum()
    .sort_values(by="total_count", ascending=False)
)

In [52]:
df_results["Top Pronoun"] = df_pronouns.sort_values(by="total_count", ascending=False)["word_lower"].head(10).values
df_results["Top Pronoun Count"] = df_pronouns.sort_values(by="total_count", ascending=False)["total_count"].head(10).values

In [54]:
df_results.head(10)
df_results.to_csv("top10_table_with_pron.csv")